# Выделим несколько самых чистых регионов без пропусков для создания дашбордов

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('df_dtp.csv', low_memory=False, parse_dates=['datetime'])

df.head()

,id,light,point,region,address,category_acc,datetime,severity,dead_count,injured_count,...,year,brand,color,model,category_veh,vehicle_age,role,gender,violations,health_status
0,189548,Светлое время суток,"{ ""lat"": 49.789873999999998, ""long"": 129.84691...",Бурейский район,"Обход п. Бурея, 5 км",Наезд на пешехода,2019-04-05 14:10:00,Легкий,0,1,...,2005.0,TOYOTA,Серый,Paseo,"С-класс (малый средний, компактный) до 4,3 м",20.0,Пешеход,Мужской,['Нахождение на проезжей части без цели её пер...,"Раненый, находящийся (находившийся) на амбула..."
1,188700,"В темное время суток, освещение отсутствует","{ ""lat"": 49.755600000000001, ""long"": 129.29220...",Бурейский район,"Благовещенск – Гомелевка, 148 км",Столкновение,2022-11-16 22:50:00,С погибшими,1,1,...,1993.0,TOYOTA,Белый,Corona,Прочие легковые автомобили,32.0,NaN,NaN,NaN,NaN
2,188700,"В темное время суток, освещение отсутствует","{ ""lat"": 49.755600000000001, ""long"": 129.29220...",Бурейский район,"Благовещенск – Гомелевка, 148 км",Столкновение,2022-11-16 22:50:00,С погибшими,1,1,...,1988.0,ISUZU,Синий,Прочие модели Isuzu,Фургоны,37.0,NaN,NaN,NaN,NaN
3,188702,"В темное время суток, освещение не включено","{ ""lat"": 49.787187000000003, ""long"": 129.81648...",Бурейский район,"Обход п. Бурея, 2 км",Съезд с дороги,2019-12-27 23:50:00,Легкий,0,1,...,2008.0,TOYOTA,Белый,Prius,"D-класс (средний) до 4,6 м",17.0,NaN,NaN,NaN,NaN
4,188704,"В темное время суток, освещение отсутствует","{ ""lat"": 49.135599999999997, ""long"": 129.1181 }",Бурейский район,"пгт Бурея, ул Райчихинская, 1",Наезд на пешехода,2016-12-30 19:10:00,С погибшими,1,0,...,1989.0,TOYOTA,Белый,Ipsum,"D-класс (средний) до 4,6 м",36.0,Пешеход,Мужской,['Нахождение на проезжей части без цели её пер...,Скончался в течение 1 суток


Выберем 20 регионов, у которых меньше всего пропусков.

In [3]:
# Считаем заполненность по каждому региону
region_quality = (
    df.drop(columns=["region"])
    .notna()
    .groupby(df["region"])
    .mean()
    .mean(axis=1)
    .reset_index(name="fill_ratio")
    .sort_values("fill_ratio", ascending=False)
)

# Оставляем только регионы с качеством > 0.9
filtered_regions = region_quality[region_quality["fill_ratio"] > 0.9]

# Берем топ-20 таких регионов
top20_regions = region_quality.head(20)

# Фильтруем датасет по этим регионам
df_filtered = df[df["region"].isin(filtered_regions["region"])]
df_top20   = df[df["region"].isin(top20_regions["region"])]

# Выводим результаты
print("Строгий фильтр >0.9:", df_filtered.shape, "регионов =", filtered_regions.shape[0])
print("Топ-20 по заполненности:", df_top20.shape, "регионов =", top20_regions.shape[0])


Строгий фильтр >0.9: (15, 26) регионов = 4
Топ-20 по заполненности: (6575, 26) регионов = 20


Подготовим данные для дашборда в Datalens.

In [4]:
df = df_top20.copy()

# Создадим удобные производные поля
df['date']  = df['datetime'].dt.date
df['year']  = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month
df['day']   = df['datetime'].dt.day
df['hour']  = df['datetime'].dt.hour

# Приведём числовые поля к числам и обработаем NaN
num_cols = ['vehicle_age', 'dead_count', 'injured_count', 'participants_count']
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

#  Сохраняем CSV для DataLens
df.to_csv("dtp_top20.csv", index=False, encoding="utf-8-sig")

**Вывод:**
- Мы взяли 20 регионов с наименьшим количеством пропусков
- Создали удобные для создания дашбордов поля, содержащие дату и время
- Привели числовые поля к числам
- Обработали пропуски
- Сохранили в CSV-файл для работы в Datalens.